## Step 1: Data Loading

In [ ]:
import torch
import torchvision.transforms as trn
import torchvision.datasets as dst

# Quick image loading setup
prep = trn.Compose([
    trn.Resize((224, 224)),
    trn.ToTensor(),
    trn.Normalize([.485, .456, .406], [.229, .224, .225])
])

ds = dst.ImageFolder("dataset/", prep)
dl = torch.utils.data.DataLoader(ds, 16, shuffle=False)


## Step2: Feature Extraction

In [ ]:
import psycopg2
import numpy as np
from tqdm import tqdm

# Fast DB setup
conn = psycopg2.connect("dbname=facedb user=postgres password=postgres")
cur = conn.cursor()
cur.execute("CREATE TABLE IF NOT EXISTS image_features (id SERIAL PRIMARY KEY, batch_id SMALLINT, img_id SMALLINT, ch SMALLINT, y SMALLINT, x SMALLINT, val REAL)")
conn.commit()

# Quick extraction function
def dump_features(dl, ratio=0.1):
    step = max(1, int(1/ratio))
    count = 0
    
    for b, (x, _) in enumerate(tqdm(dl)):
        data = []
        x = x.numpy()
        
        # Nested loops replaced with vectorized operations where possible
        for i in range(x.shape[0]):
            for c in range(3):
                pixels = [(b, i, c, y, z, round(float(x[i,c,y,z]), 4)) 
                         for y in range(0, 224, step) 
                         for z in range(0, 224, step)
                         if abs(x[i,c,y,z]) > 0.001]
                data.extend(pixels)
                
        if data:
            args = ','.join(cur.mogrify("(%s,%s,%s,%s,%s,%s)", i).decode() for i in data)
            cur.execute(f"INSERT INTO image_features (batch_id,img_id,ch,y,x,val) VALUES {args}")
            conn.commit()
            count += len(data)
    
    return count

n = dump_features(data_loader, 0.1)
print(f"{n} pixels stored")
cur.close()
conn.close()


## Step 3: Model Profiling

In [ ]:
import onnx, time, numpy as np
from onnxruntime import InferenceSession

m = onnx.load("resnext50.onnx") 
s = InferenceSession("resnext50.onnx")

x = np.random.rand(1, 3, 224, 224).astype('float32')
n = s.get_inputs()[0].name

 print(f"Parameters: {len(m.graph.initializer)}, Time: {ms:.2f}ms")


## Step 4: Data Preparation

In [ ]:
import onnx
import time
import numpy as np
import psycopg2
from psycopg2.extras import execute_batch
from contextlib import contextmanager

class ModelDBTransformer:
    """Transform ONNX model parameters into relational database schema"""
    
    def __init__(self, conn_str):
        self.conn_str = conn_str
        self.cleanup_tables = set()
        
    @contextmanager
    def _db_cursor(self):
        """Context manager for database connections"""
        conn = psycopg2.connect(self.conn_str)
        try:
            with conn.cursor() as cur:
                yield cur
                conn.commit()
        except Exception as e:
            conn.rollback()
            raise e
        finally:
            conn.close()
    
    def _execute_sql(self, query, params=None, batch_data=None):
        """Unified SQL execution with error handling"""
        with self._db_cursor() as cur:
            if batch_data:
                execute_batch(cur, query, batch_data)
            elif params:
                cur.execute(query, params)
            else:
                cur.execute(query)
    
    def analyze_model(self, model_path):
        """Parse ONNX model structure"""
        model = onnx.load(model_path)
        print(f"Model Analysis:\n"
              f"• Layers: {len(model.graph.node)}\n"
              f"• Parameters: {len(model.graph.initializer)}")
        return model
    
    def _generate_table_name(self, tensor_name):
        """Generate simplified table name from tensor name"""
        return f"param_{tensor_name.split('/')[-1].replace('.', '_')[:30]}"
    
    def extract_parameters(self, model):
        """Convert model parameters to database tables"""
        param_tables = {}
        
        for tensor in model.graph.initializer:
            if tensor.data_type != 1:  # Skip non-float32 tensors
                continue
                
            table_name = self._generate_table_name(tensor.name)
            array = np.frombuffer(tensor.raw_data, dtype=np.float32).reshape(tensor.dims)
            param_tables[table_name] = array
            
            if len(array.shape) > 1:
                print(f"Parameter: {tensor.name[:20]} → Table: {table_name} {array.shape}")
                
        return param_tables
    
    def _create_param_table(self, table_name, shape):
        """Dynamic table creation based on tensor dimensions"""
        schema = {
            2: ("(unit_out SMALLINT, unit_in SMALLINT, value REAL)",
                "(%s, %s, %s)"),
            4: ("(filter_out SMALLINT, filter_in SMALLINT, h SMALLINT, w SMALLINT, value REAL)",
                "(%s, %s, %s, %s, %s)")
        }
        
        dim_type = 2 if len(shape) == 2 else 4  # Handle 2D/4D tensors
        columns, placeholders = schema[dim_type]
        
        self._execute_sql(f"CREATE TABLE IF NOT EXISTS {table_name} {columns}")
        self.cleanup_tables.add(table_name)
        return placeholders
    
    def store_parameters(self, parameters):
        """Batch insert parameters into database"""
        for table_name, tensor in parameters.items():
            placeholders = self._create_param_table(table_name, tensor.shape)
            
            # Generate insert data based on tensor shape
            if len(tensor.shape) == 2:
                data = [(i, j, float(tensor[i, j])) 
                       for i in range(tensor.shape[0]) 
                       for j in range(tensor.shape[1])]
            else:
                data = [(i, j, h, w, float(tensor[i, j, h, w]))
                       for i in range(tensor.shape[0])
                       for j in range(tensor.shape[1])
                       for h in range(tensor.shape[2])
                       for w in range(tensor.shape[3])]

            self._execute_sql(
                f"INSERT INTO {table_name} VALUES {placeholders}",
                batch_data=data
            )
    
    def create_conv_mapping(self, base_name, input_size, kernel, stride, pad, channels):
        """Generate convolution operation mapping table"""
        table_name = f"conv_{input_size}_{kernel}_{stride}_{channels}"
        if table_name in self.cleanup_tables:
            return
            
        self._execute_sql(f"""
            CREATE TABLE {table_name} (
                channel SMALLINT,
                input_idx SMALLINT,
                output_idx SMALLINT,
                kernel_idx SMALLINT
            )
        """)
        
        # Generate mapping indices
        mappings = []
        output_pos = 0
        padded_size = input_size + 2*pad
        
        for y in range(0, padded_size - kernel + 1, stride):
            for x in range(0, padded_size - kernel + 1, stride):
                kernel_pos = 0
                for c in range(channels):
                    for dy in range(kernel):
                        for dx in range(kernel):
                            y_in = y + dy
                            x_in = x + dx
                            if pad <= y_in < input_size + pad and pad <= x_in < input_size + pad:
                                input_pos = (y_in - pad)*input_size + (x_in - pad)
                                mappings.append((c, input_pos, output_pos, kernel_pos))
                            kernel_pos += 1
                output_pos += 1
                
        self._execute_sql(
            f"INSERT INTO {table_name} VALUES (%s, %s, %s, %s)",
            batch_data=mappings
        )
        self.cleanup_tables.add(table_name)
    
    def transform(self, model_path):
        """Main transformation pipeline"""
        start = time.time()
        
        print(f"Processing: {model_path}")
        model = self.analyze_model(model_path)
        parameters = self.extract_parameters(model)
        self.store_parameters(parameters)
        
        # Create standard convolution mappings
        self.create_conv_mapping("input", 224, 7, 2, 3, 3)
        self.create_conv_mapping("block1", 56, 3, 1, 1, 64)
        
        # Generate cleanup script
        with open("model_cleanup.sql", "w") as f:
            f.writelines(f"DROP TABLE IF EXISTS {tbl};\n" for tbl in self.cleanup_tables)
            
        print(f"Completed in {time.time()-start:.2f}s")

# Usage
transformer = ModelDBTransformer("dbname=facedb user=postgres password=postgres host=localhost")
transformer.transform("resnetx50.onnx")

## Step 5: Query Composition

In [ ]:
WITH
Conv201_fwd0 AS (
    select batch_id, K.unit_out, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM input_feature_map F INNER JOIN param_Conv201_weight K
        on F.order_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.matrix_id
),
Relu203_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv201_fwd0 F
),
Relu203_fwd0_mapped AS (
    select batch_id, F.kernel_id, matrix_id, order_id, value
        FROM Relu203_fwd0 F INNER JOIN conv_224_7_2_3 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
MaxPool205_fwd0 AS (
    select batch_id, kernel_id, matrix_id as tuple_id, max(value) as value
        FROM Relu203_fwd0_mapped
    GROUP BY batch_id, kernel_id, matrix_id
),
Conv206_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM MaxPool205_fwd0 F INNER JOIN param_Conv206_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Conv207_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM MaxPool205_fwd0 F INNER JOIN param_Conv207_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu208_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv206_fwd0 F
),
Relu208_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu208_fwd0 F INNER JOIN conv_56_3_1_64 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv209_fwd0 AS (
    select batch_id, (F.gn*4+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu208_fwd0_mapped F INNER JOIN param_Conv209_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu210_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv209_fwd0 F
),
Conv211_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu210_fwd0 F INNER JOIN param_Conv211_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add212_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Conv207_fwd0 A INNER JOIN Conv211_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu213_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add212_fwd0 F
),
Conv214_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu213_fwd0 F INNER JOIN param_Conv214_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu215_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv214_fwd0 F
),
Relu215_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu215_fwd0 F INNER JOIN conv_56_3_1_64 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv216_fwd0 AS (
    select batch_id, (F.gn*4+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu215_fwd0_mapped F INNER JOIN param_Conv216_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu217_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv216_fwd0 F
),
Conv218_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu217_fwd0 F INNER JOIN param_Conv218_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add219_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu213_fwd0 A INNER JOIN Conv218_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu220_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add219_fwd0 F
),
Conv221_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu220_fwd0 F INNER JOIN param_Conv221_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu222_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv221_fwd0 F
),
Relu222_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu222_fwd0 F INNER JOIN conv_56_3_1_64 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv223_fwd0 AS (
    select batch_id, (F.gn*4+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu222_fwd0_mapped F INNER JOIN param_Conv223_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu224_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv223_fwd0 F
),
Conv225_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu224_fwd0 F INNER JOIN param_Conv225_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add226_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu220_fwd0 A INNER JOIN Conv225_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu227_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add226_fwd0 F
),
Conv228_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu227_fwd0 F INNER JOIN param_Conv228_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu229_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv228_fwd0 F
),
Relu229_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu229_fwd0 F INNER JOIN conv_56_3_1_64 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv230_fwd0 AS (
    select batch_id, (F.gn*4+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu229_fwd0_mapped F INNER JOIN param_Conv230_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu231_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv230_fwd0 F
),
Conv232_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu231_fwd0 F INNER JOIN param_Conv232_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add233_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu227_fwd0 A INNER JOIN Conv232_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu234_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add233_fwd0 F
),
Conv235_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu234_fwd0 F INNER JOIN param_Conv235_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu236_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv235_fwd0 F
),
Relu236_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu236_fwd0 F INNER JOIN conv_56_3_1_64 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv237_fwd0 AS (
    select batch_id, (F.gn*4+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu236_fwd0_mapped F INNER JOIN param_Conv237_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu238_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv237_fwd0 F
),
Conv239_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu238_fwd0 F INNER JOIN param_Conv239_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add240_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu234_fwd0 A INNER JOIN Conv239_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu241_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add240_fwd0 F
),
Conv242_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu241_fwd0 F INNER JOIN param_Conv242_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu241_fwd0_mapped AS (
    select batch_id, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu241_fwd0 F INNER JOIN conv_56_3_1_64 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv243_fwd0 AS (
    select batch_id, K.unit_out, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu241_fwd0_mapped F INNER JOIN param_Conv243_weight K
        on F.order_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.matrix_id
),
Relu244_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv242_fwd0 F
),
Relu244_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu244_fwd0 F INNER JOIN conv_56_3_1_64 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv245_fwd0 AS (
    select batch_id, (F.gn*8+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu244_fwd0_mapped F INNER JOIN param_Conv245_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu246_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv245_fwd0 F
),
Conv247_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu246_fwd0 F INNER JOIN param_Conv247_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add248_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Conv243_fwd0 A INNER JOIN Conv247_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu249_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add248_fwd0 F
),
Conv250_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu249_fwd0 F INNER JOIN param_Conv250_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu251_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv250_fwd0 F
),
Relu251_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu251_fwd0 F INNER JOIN conv_28_3_1_128 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv252_fwd0 AS (
    select batch_id, (F.gn*8+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu251_fwd0_mapped F INNER JOIN param_Conv252_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu253_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv252_fwd0 F
),
Conv254_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu253_fwd0 F INNER JOIN param_Conv254_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add255_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu249_fwd0 A INNER JOIN Conv254_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu256_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add255_fwd0 F
),
Conv257_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu256_fwd0 F INNER JOIN param_Conv257_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu258_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv257_fwd0 F
),
Relu258_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu258_fwd0 F INNER JOIN conv_28_3_1_128 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv259_fwd0 AS (
    select batch_id, (F.gn*8+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu258_fwd0_mapped F INNER JOIN param_Conv259_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu260_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv259_fwd0 F
),
Conv261_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu260_fwd0 F INNER JOIN param_Conv261_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add262_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu256_fwd0 A INNER JOIN Conv261_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu263_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add262_fwd0 F
),
Conv264_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu263_fwd0 F INNER JOIN param_Conv264_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu265_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv264_fwd0 F
),
Relu265_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu265_fwd0 F INNER JOIN conv_28_3_1_128 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv266_fwd0 AS (
    select batch_id, (F.gn*8+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu265_fwd0_mapped F INNER JOIN param_Conv266_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu267_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv266_fwd0 F
),
Conv268_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu267_fwd0 F INNER JOIN param_Conv268_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add269_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu263_fwd0 A INNER JOIN Conv268_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu270_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add269_fwd0 F
),
Conv271_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu270_fwd0 F INNER JOIN param_Conv271_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu272_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv271_fwd0 F
),
Relu272_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu272_fwd0 F INNER JOIN conv_14_3_1_256 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv273_fwd0 AS (
    select batch_id, (F.gn*8+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu272_fwd0_mapped F INNER JOIN param_Conv273_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu274_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv273_fwd0 F
),
Conv275_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu274_fwd0 F INNER JOIN param_Conv275_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add276_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu270_fwd0 A INNER JOIN Conv275_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu277_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add276_fwd0 F
),
Conv278_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu277_fwd0 F INNER JOIN param_Conv278_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu277_fwd0_mapped AS (
    select batch_id, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu277_fwd0 F INNER JOIN conv_14_3_1_256 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv279_fwd0 AS (
    select batch_id, K.unit_out, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu277_fwd0_mapped F INNER JOIN param_Conv279_weight K
        on F.order_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.matrix_id
),
Relu280_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv278_fwd0 F
),
Relu280_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu280_fwd0 F INNER JOIN conv_14_3_1_256 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv281_fwd0 AS (
    select batch_id, (F.gn*16+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu280_fwd0_mapped F INNER JOIN param_Conv281_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu282_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv281_fwd0 F
),
Conv283_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu282_fwd0 F INNER JOIN param_Conv283_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add284_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Conv279_fwd0 A INNER JOIN Conv283_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu285_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add284_fwd0 F
),
Conv286_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu285_fwd0 F INNER JOIN param_Conv286_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu287_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv286_fwd0 F
),
Relu287_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu287_fwd0 F INNER JOIN conv_14_3_1_256 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv288_fwd0 AS (
    select batch_id, (F.gn*16+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu287_fwd0_mapped F INNER JOIN param_Conv288_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu289_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv288_fwd0 F
),
Conv290_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu289_fwd0 F INNER JOIN param_Conv290_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add291_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu285_fwd0 A INNER JOIN Conv290_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu292_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add291_fwd0 F
),
Conv293_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu292_fwd0 F INNER JOIN param_Conv293_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu294_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv293_fwd0 F
),
Relu294_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu294_fwd0 F INNER JOIN conv_7_3_1_512 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv295_fwd0 AS (
    select batch_id, (F.gn*16+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu294_fwd0_mapped F INNER JOIN param_Conv295_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu296_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv295_fwd0 F
),
Conv297_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu296_fwd0 F INNER JOIN param_Conv297_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add298_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu292_fwd0 A INNER JOIN Conv297_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu299_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add298_fwd0 F
),
Conv300_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu299_fwd0 F INNER JOIN param_Conv300_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu301_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv300_fwd0 F
),
Relu301_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu301_fwd0 F INNER JOIN conv_7_3_1_512 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv302_fwd0 AS (
    select batch_id, (F.gn*16+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu301_fwd0_mapped F INNER JOIN param_Conv302_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu303_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv302_fwd0 F
),
Conv304_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu303_fwd0 F INNER JOIN param_Conv304_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add305_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu299_fwd0 A INNER JOIN Conv304_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu306_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add305_fwd0 F
),
Conv307_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu306_fwd0 F INNER JOIN param_Conv307_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Relu308_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv307_fwd0 F
),
Relu308_fwd0_mapped AS (
    select batch_id, M.channel as gn, M.output_idx as matrix_id, M.kernel_idx as order_id, value
        FROM Relu308_fwd0 F INNER JOIN conv_7_3_1_512 M
        on F.kernel_id = M.channel and F.tuple_id = M.input_idx
),
Conv309_fwd0 AS (
    select batch_id, (F.gn*16+K.filter_out) as kernel_id, F.matrix_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu308_fwd0_mapped F INNER JOIN param_Conv309_weight K
        on F.order_id=K.unit_in and F.gn=K.filter_in
    GROUP BY F.batch_id, F.gn, K.filter_out, F.matrix_id
),
Relu310_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Conv309_fwd0 F
),
Conv311_fwd0 AS (
    select batch_id, K.unit_out, F.tuple_id as tuple_id,
    sum(F.value*K.value) as value
        FROM Relu310_fwd0 F INNER JOIN param_Conv311_weight K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out, F.tuple_id
),
Add312_fwd0 AS (
    select A.batch_id, A.kernel_id, A.tuple_id,
    A.value + B.value as value
        FROM Relu306_fwd0 A INNER JOIN Conv311_fwd0 B
        on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
        and A.tuple_id=B.tuple_id
),
Relu313_fwd0 AS (
    select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
        FROM Add312_fwd0 F
),
AveragePool315_fwd0 AS (
    select batch_id, kernel_id, avg(value) as value
        FROM Relu313_fwd0
    GROUP BY batch_id, kernel_id
),
MatMul318_fwd0 AS (
    select batch_id, K.unit_out,
    sum(F.value*K.value) as value
        FROM AveragePool315_fwd0 F INNER JOIN param_fc_weights K
        on F.kernel_id=K.unit_in
    GROUP BY F.batch_id, K.unit_out
),
SELECT l.name AS res FROM
cifar10_labels l
JOIN (
    SELECT DISTINCT on (batch_id) batch_id, kernel_id+1 AS label
    FROM MatMul318_fwd0 ORDER BY batch_id, value DESC) t
    ON t.label = l.label
